In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')  # 切换到 Qt5Agg 后端
# Use pandas to read the content of the log0-raw.log file
data = pd.read_csv('log0-raw.log', sep=' ', header=None)
t1 = data[0]-data[0][0]


In [8]:
k = 4
plt.plot(t1,data[20+k]-data[0+k])
plt.legend()
plt.show()


C:\Users\18306\AppData\Local\Temp\ipykernel_32508\1295820491.py:3: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


In [42]:
import pandas as pd
from sliding_window_kmeans import sliding_window_kmeans
from sklearn import metrics
# 示例数据
data[25] = pd.to_numeric(data[25], errors='coerce')


# 调用函数
labels, plot_data, rolling_variance, cluster_ranges = sliding_window_kmeans(
    data, column=26, window_size=50, n_clusters=4, remove_outliers=True,plot=True
)
print(metrics.calinski_harabasz_score(plot_data.values.reshape(-1, 1), labels))
# 输出每个聚类的横坐标范围
print("Cluster Ranges:", sorted(cluster_ranges, key=lambda x: x[0]))

289347.74854949967
Cluster Ranges: [(0, 31697), (31698, 63528), (63529, 95748), (95749, 128223)]


In [41]:
index1 = cluster_ranges[0][0]
index2 = t1[t1>220].index[0]
plt.plot(t1[index1:index2], data[25][index1:index2])
plt.show()

In [70]:
#线性回归
from sklearn.linear_model import LinearRegression
#创建线性回归模型
linear_model = LinearRegression()
#采用部分点进行训练
#numpy提取部分数据
[x,y] = [20,24]
train_data = data[y][index1:index2]
train_t1 = data[x][index1:index2]
#将数据转换为二维数组
train_t1 = np.array(train_t1).reshape(-1, 1)
train_data = np.array(train_data).reshape(-1, 1)
plt.plot(train_t1, train_data, 'o', markersize=1.5)
plt.show()

In [71]:
#训练模型
linear_model.fit(train_t1, train_data)
#绘制拟合直线和原始数据
#根据模型参数绘制拟合直线
pre_data = linear_model.predict(np.array(data[x]).reshape(-1, 1))
plt.plot(data[x], data[y], 'o', markersize=1)
plt.plot(data[x], pre_data, 'r', label='y=0.0001x+0.0001', color='red', linewidth=2)
plt.show()
#输出模型参数
#输出模型评分

delay_p_offset = linear_model.coef_[0][0]
skew = linear_model.intercept_[0]
print("delay_p_offset:", delay_p_offset)
print("skew:", skew)

C:\Users\18306\AppData\Local\Temp\ipykernel_9572\2015553714.py:7: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "r" (-> color=(1.0, 0.0, 0.0, 1)). The keyword argument will take precedence.
  plt.plot(data[x], pre_data, 'r', label='y=0.0001x+0.0001', color='red', linewidth=2)


delay_p_offset: -1.8449748924781758e-06
skew: 0.3086155094704866


In [7]:
data25 = pd.Series(data[25])  # Convert to pandas Series
jitter = pd.Series(pre_data_25.flatten() - data25.values)  # Ensure both are Series
jitter_df = pd.DataFrame(jitter, columns=["Jitter"])  # Convert to DataFrame with a column name
labels, plot_data, rolling_variance, cluster_ranges = sliding_window_kmeans(
    jitter_df.reset_index(), column="Jitter", window_size=50, n_clusters=4, remove_outliers=True, plot=True
)
print(metrics.calinski_harabasz_score(plot_data.values.reshape(-1, 1), labels))
# 输出每个聚类的横坐标范围
print("Cluster Ranges:", sorted(cluster_ranges, key=lambda x: x[0]))



KeyError: 0

In [44]:
#计算端到端延迟抖动
delay_mean = data[25]
delay_mean = delay_mean.values.reshape(-1, 1)
print(predict_delay.shape)
print(delay_mean.shape)
delay_jitter = predict_delay - delay_mean

delay_jitter = delay_jitter*10e6
print(delay_jitter[:10])

(128224, 1)
(128224, 1)
[[ 564.76361804]
 [-380.75241259]
 [-323.67842556]
 [-434.19441204]
 [-424.53037202]
 [-453.00628784]
 [-434.27224341]
 [-394.59819897]
 [-445.11414129]
 [-350.63007036]]


In [47]:
import seaborn as sns
# 使用 histplot 并设置纵轴为密度
sns.histplot(delay_jitter, bins=100, kde=False, stat="density")
plt.xlabel("Delay Jitter")
plt.ylabel("Cumulative Density")
plt.title("Cumulative Distribution Function (CDF)")
plt.show()

In [66]:
from scipy.stats import expon

# 拟合指数分布
params = expon.fit(delay_jitter)  # 返回参数 (loc, scale)
loc, scale = params
print(f"loc: {loc}, scale: {scale}")
# 绘制直方图
sns.histplot(delay_jitter, bins=100, kde=False, stat="density", label="Data")

# 绘制拟合的指数分布曲线
x = np.linspace(min(delay_jitter), max(delay_jitter), 1000)
pdf_expon = expon.pdf(x, loc=loc, scale=scale)  # 计算指数分布的概率密度函数
plt.plot(x, pdf_expon, 'r-', label=f"Exponential Fit (λ={1/scale:.6f})")

# 添加图例和标签
plt.xlabel("Delay Jitter")
plt.ylabel("Density")
plt.title("Exponential Distribution Fit")
plt.legend()
plt.show()

NameError: name 'delay_jitter' is not defined